# Session 1, Step 1: Stryker's share prices

**Goal:** download 3 years of Stryker (SYK) daily share prices, look at them, save them, and draw a chart.

Run the cells **one at a time, from top to bottom** (click a cell, then `Shift + Enter`). Read the text above each cell *before* running it.

## Cell 1: load the tools

`import` loads a library (a toolbox someone else wrote) so we can use it.

- `pandas` (nickname `pd`): tables of data, like Excel inside Python.
- `yfinance` (nickname `yf`): downloads data from Yahoo Finance.
- `matplotlib.pyplot` (nickname `plt`): draws charts.
- `Path`: handles file and folder paths.

Nothing visible happens when you run this. That's normal: no output means no error.

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from pathlib import Path

## Cell 2: the settings

We write our choices into **variables** (named boxes holding a value), so they're in one place and easy to change.

- `TICKER`: the company's stock-market code. Stryker = `SYK`.
- `START` / `END`: the 3-year window. Yahoo's `end` date is **not included**, so to include 6 November we write 7 November.
- `RAW_DIR`: the folder where raw downloads are saved. `..` means "go up one folder" (from `notebooks/` to the project folder).

Names in CAPITALS are a Python habit for settings that never change.

In [ ]:
TICKER = "SYK"
START = "2023-11-06"
END = "2026-11-07"   # not included, so the last day is 6 Nov 2026
RAW_DIR = Path("..") / "data" / "raw"

print("Company:", TICKER)
print("From", START, "to", END)

## Cell 3: download the prices

`yf.download(...)` asks Yahoo Finance for the data and gives back a **DataFrame** (a table). We store it in a variable called `prices`.

- `auto_adjust=True`: adjust old prices for **dividends and stock splits**. Without this, a 2-for-1 split would look like the price crashing 50% overnight, and returns would be wrong.
- `progress=False`: hide the download progress bar.
- `multi_level_index=False`: keep the column names simple (`Close`, not `(Close, SYK)`).

In [ ]:
prices = yf.download(TICKER, start=START, end=END, auto_adjust=True,
                     progress=False, multi_level_index=False)

print("Downloaded", len(prices), "days of prices")

## Cell 4: look at the data

**Always look at data before using it.** These commands show:

- `.head()`: the first 5 rows.
- `.tail()`: the last 5 rows.

Each row is one trading day. The columns:

| Column | Meaning |
|---|---|
| Open | price at the start of the day |
| High / Low | highest and lowest price during the day |
| Close | price at the end of the day (**the one we'll use**) |
| Volume | how many shares changed hands |

**Questions for you** (answer them before Cell 5):
1. What was Stryker's closing price on the first day?
2. And on the last day?
3. About 252 trading days a year × 3 years ≈ 756. How many rows did you get? Why aren't there 365 × 3 = 1,095?

In [ ]:
print(prices.head())
print()
print(prices.tail())

## Cell 5: save the raw data

We save exactly what we downloaded, untouched, into `data/raw/`. Why?

- **Reproducibility:** Yahoo's data can change or the service can go down. With a saved copy, the analysis always gives the same answer.
- **The data rule in our README:** record *when* and *where* data came from. The date of the download goes into the file name.

`pd.Timestamp.today()` is today's date; `.strftime("%Y-%m-%d")` formats it as text like `2026-11-06`. The `f"..."` is an **f-string**: anything in `{ }` gets replaced by its value.

In [ ]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")
file_path = RAW_DIR / f"{TICKER}_prices_downloaded_{today}.csv"

prices.to_csv(file_path)
print("Saved to", file_path)

## Cell 6: draw the chart

- `plt.subplots(...)` creates an empty chart (`fig` is the whole picture, `ax` is the plot area). `figsize` is the width and height in inches.
- `prices["Close"]` picks the **Close** column.
- `ax.plot(...)` draws it as a line: dates on the x-axis, prices on the y-axis.
- The `set_...` lines add a title and axis labels.

**Question for you:** looking at the chart, describe Stryker's last 3 years in one sentence. Was there a big fall? When?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(prices.index, prices["Close"])
ax.set_title("Stryker (SYK): daily closing price, adjusted")
ax.set_xlabel("Date")
ax.set_ylabel("Price (US$)")
plt.show()